# Experiment 034: W2 smoke qualification (Colab)

The session uses only the disjoint smoke facts. It checks rank-16 LoRA can
teach the fictional associations and measures the SFT and GRPO rates before the
real W2 design is fixed. No MMLU evaluation or registered W2 fact is used.


## 0. Compute units and mount Drive

In [2]:
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 522.26
unit_rate = 6.77
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])
    if not os.path.exists(open_path):
        record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                  'notebook': '034_colab_w2_smoke', 'colab_cu_before': cu_before,
                  'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(record, f, indent=2)
        print('saved open session:', session_id)


session 20260815_190425 | cu_before = 522.26
Mounted at /content/drive
saved open session: 20260815_190425


## 1. GPU check: A100 80GB


In [3]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the A100 80GB runtime'


GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-b3c35398-aa44-74cb-564d-c433bb1d32fd)
total vram: 79.3 GiB


## 2. Clone or update the repo


In [4]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies


In [5]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
print('dependencies installed')


dependencies installed


## 4. W&B key

The Colab secret is used first. The prompt is only a fallback.


In [6]:
import getpass, os
wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None
if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')
os.environ['WANDB_API_KEY'] = wkey
os.environ['WANDB_PROJECT'] = 'inm363-grpo'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
wkey = None
print('W&B key loaded')


W&B key loaded


## 5. Session setup


In [7]:
import json, os, shutil, sys
os.chdir(code_root)
sys.path.insert(0, os.getcwd())

from config import load_config, run_tag
from data.synthetic_facts import build_bundle
from file_utils import read_json
from obs import append_ledger, gpu_snapshot
from scripts.grpo_worker import stream_subprocess
from scripts.s12_eval import atomic_write_json, read_jsonl
from scripts.s28_compose import IntegrityError, adapter_hashes_of
from scripts.w2_probe import (bind_start, commit_eval, eval_state, load_bundle,
                              smoke_qualification, verify_start, write_bundle)

SFT_CONFIG = 'configs/034_smoke_sft_colab_b15.yaml'
GRPO_CONFIG = 'configs/034_smoke_grpo_colab_b15.yaml'
SFT_CFG = load_config(SFT_CONFIG)
GRPO_CFG = load_config(GRPO_CONFIG)
assert SFT_CFG.log.wandb_project == GRPO_CFG.log.wandb_project == 'inm363-grpo'
assert os.environ['WANDB_PROJECT'] == 'inm363-grpo'
assert cu_before is not None and unit_rate is not None, 'fill section 0 first'

SMOKE_ROOT = os.path.join(RESULTS, '034_smoke')
BUNDLE_PATH = os.path.join(SMOKE_ROOT, 'smoke_bundle.json')
START_RECORD = os.path.join(SMOKE_ROOT, 'start_seed0.json')
PARENT_TAG = '009_COLAB_FULL_B15_sft_grpo_seed0'
PARENT_MANIFEST = os.path.join(RESULTS, PARENT_TAG, 'manifest.json')
BASE_RESULT = os.path.join(RESULTS, '034_smoke_baseline_seed0')
SFT_TAG, GRPO_TAG = run_tag(SFT_CFG), run_tag(GRPO_CFG)
SFT_RESULT, GRPO_RESULT = os.path.join(RESULTS, SFT_TAG), os.path.join(RESULTS, GRPO_TAG)
SFT_ADAPTER = os.path.join(ROOT, 'checkpoints', SFT_TAG, 'sft')
GRPO_ADAPTER = os.path.join(ROOT, 'checkpoints', GRPO_TAG, 'grpo')
LEDGER = os.path.join(RESULTS, 'compute_ledger.jsonl')
os.makedirs(SMOKE_ROOT, exist_ok=True)

free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
assert free_gb >= 20, 'need at least 20GB free on Drive'
print('SFT:', SFT_TAG)
print('GRPO:', GRPO_TAG)


drive free: 178 GB
SFT: 034_smoke_COLAB_FULL_B15_sft_only_seed0
GRPO: 034_smoke_COLAB_FULL_B15_external_sft_grpo_seed0


## 6. Wiring checks


In [8]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_training_wrapper_contracts.py',
    'tests/test_synthetic_facts.py',
    'tests/test_row_evaluation.py',
    'tests/test_w2_probe.py',
    'tests/test_w2_worker.py',
    'tests/test_protocol_freeze.py',
], check=True)
print('wiring passed')


wiring passed


## 7. Build the disjoint smoke data and bind the start


In [9]:
smoke = build_bundle('smoke', seed=34001, n_injected=16, n_control=16)
write_bundle(BUNDLE_PATH, smoke)
assert load_bundle(BUNDLE_PATH) == smoke

start = bind_start(SFT_CFG, BUNDLE_PATH, PARENT_MANIFEST, PARENT_TAG, START_RECORD)
assert verify_start(GRPO_CFG, START_RECORD) == start
assert start['dataset_sha256'] == smoke['sha256']
print('dataset:', smoke['sha256'])
print('start:', start['merged_dir'])


dataset: a9d359dfe03251b0b08e6d25f22870348dc900a7d4c505cb3b341a1b92ee1583
start: /content/drive/MyDrive/inm363/checkpoints/009_COLAB_FULL_B15_sft_grpo_seed0/sft_merged


## 8. Worker helpers


In [10]:
import json, os, shutil, subprocess, sys, time

def free_vram_gib():
    value = subprocess.check_output([
        'nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'
    ], text=True).strip().splitlines()[0]
    return float(value) / 1024

def run_child(cmd, status_path, stage, tag, train_steps=0):
    if os.path.exists(status_path):
        old = read_json(status_path)
        if old.get('completed') is not True and old.get('error_type') not in (None, 'KeyboardInterrupt'):
            raise RuntimeError(f'{stage}: previous worker failure needs review')
        os.remove(status_path)
    started = time.time()
    completed = False
    status = {}
    try:
        free_gib = free_vram_gib()
        print(f'free vram before {stage}: {free_gib:.1f} GiB')
        assert free_gib >= 60, 'restart the runtime before launching the next worker'
        proc = stream_subprocess(cmd)
        if os.path.exists(status_path):
            status = read_json(status_path)
        assert proc.returncode == 0, f'{stage}: worker returned {proc.returncode}'
        assert status.get('completed') is True, f'{stage}: worker did not complete'
        completed = True
        return status
    finally:
        elapsed = round(time.time() - started, 1)
        gpu_row = status.get('gpu', gpu_snapshot())
        if status.get('peak_vram_gib') is not None:
            gpu_row['peak_vram_gb'] = status['peak_vram_gib']
        steps = status.get('steps_this_attempt', train_steps if completed else None)
        append_ledger(LEDGER, {'run_tag': tag, 'condition': stage,
                               'timings_sec': {stage: elapsed}, 'train_steps': steps,
                               'completed': completed, 'gpu': gpu_row})

def train_smoke(branch, cfg_path, tag, adapter_dir, result_dir):
    status_path = os.path.join(result_dir, f'{branch}_train_status.json')
    try:
        adapter_hashes_of(adapter_dir)
        trained = os.path.exists(status_path) and read_json(status_path).get('completed') is True
    except Exception:
        trained = False
    if trained:
        print(branch, 'adapter already trained')
        return read_json(status_path)
    os.makedirs(result_dir, exist_ok=True)
    cmd = [sys.executable, '-u', 'scripts/w2_worker.py',
           '--task', 'train', '--branch', branch, '--config', cfg_path,
           '--bundle', BUNDLE_PATH, '--start-record', START_RECORD,
           '--status', status_path, '--out-dir', adapter_dir,
           '--log-dir', result_dir, '--run-name', f'{tag}_{branch}',
           '--report-to', 'tensorboard', 'wandb']
    cfg = load_config(cfg_path)
    return run_child(cmd, status_path, f'W2_smoke_{branch}_train', tag,
                     train_steps=cfg.train.max_steps)

def evaluate_smoke(branch, cfg_path, tag, adapter_dir, result_dir):
    state = eval_state(result_dir, branch, adapter_dir, BUNDLE_PATH, START_RECORD)
    if state == 'tampered':
        raise IntegrityError(f'{branch}: committed evaluation is inconsistent')
    if state == 'committed':
        print(branch, 'evaluation already committed')
        return
    scratch = f'/content/w2_smoke_{branch}_{session_id}'
    shutil.rmtree(scratch, ignore_errors=True)
    status_path = os.path.join(result_dir, f'{branch}_eval_status.json')
    cmd = [sys.executable, '-u', 'scripts/w2_worker.py',
           '--task', 'eval', '--branch', branch, '--config', cfg_path,
           '--bundle', BUNDLE_PATH, '--start-record', START_RECORD,
           '--status', status_path, '--scratch-dir', scratch]
    if adapter_dir:
        cmd.extend(['--adapter-dir', adapter_dir])
    status = run_child(cmd, status_path, f'W2_smoke_{branch}_eval', tag)
    commit_eval(result_dir, scratch, branch, BUNDLE_PATH, START_RECORD,
                adapter_dir=adapter_dir, extra={'worker_status': status})
    shutil.rmtree(scratch, ignore_errors=True)


## 9. Shared baseline


In [11]:
evaluate_smoke('baseline', SFT_CONFIG, '034_smoke_baseline_seed0', None, BASE_RESULT)


free vram before W2_smoke_baseline_eval: 79.2 GiB
Loading weights: 100% 338/338 [00:00<00:00, 2452.08it/s]
baseline eval (96 items): 100% 6/6 [01:03<00:00, 10.62s/it, acc 0.260 fmt 0.958]
Loading weights: 100% 338/338 [00:00<00:00, 2266.47it/s]
baseline logprob (96 items): 100% 6/6 [00:00<00:00, 22.16it/s, acc 0.281]


## 10. Supervised smoke branch


In [12]:
sft_status = train_smoke('sft', SFT_CONFIG, SFT_TAG, SFT_ADAPTER, SFT_RESULT)
evaluate_smoke('sft', SFT_CONFIG, SFT_TAG, SFT_ADAPTER, SFT_RESULT)
print('SFT wall seconds:', sft_status.get('wall_seconds'))


free vram before W2_smoke_sft_train: 79.2 GiB
Loading weights: 100% 338/338 [00:02<00:00, 164.74it/s]
Adding EOS to train dataset: 100% 16/16 [00:00<00:00, 3825.61 examples/s]
Tokenizing train dataset: 100% 16/16 [00:00<00:00, 1570.61 examples/s]
Building labels for train dataset: 100% 16/16 [00:00<00:00, 3411.22 examples/s]
2026-08-15 19:31:00.186090: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-15 19:31:00.253404: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
The tokenizer has new PAD/BOS/EOS tokens that d

## 11. Qualification gate


In [13]:
base_gen = read_jsonl(os.path.join(BASE_RESULT, 'baseline_preds.jsonl'))
base_probe = read_jsonl(os.path.join(BASE_RESULT, 'baseline_logprob_preds.jsonl'))
sft_gen = read_jsonl(os.path.join(SFT_RESULT, 'sft_preds.jsonl'))
sft_probe = read_jsonl(os.path.join(SFT_RESULT, 'sft_logprob_preds.jsonl'))
qualification = smoke_qualification(base_gen, base_probe, sft_gen, sft_probe)
qualification.update({'dataset_sha256': smoke['sha256'],
                      'start_record': START_RECORD, 'sft_config': SFT_CONFIG})
qualification_path = os.path.join(SMOKE_ROOT, 'sft_qualification.json')
if os.path.exists(qualification_path):
    assert read_json(qualification_path) == qualification, 'qualification result changed on rerun'
else:
    atomic_write_json(qualification_path, qualification)
print(json.dumps(qualification, indent=2))
assert qualification['passed'] is True, 'LoRA qualification failed, stop before the GRPO smoke'


{
  "passed": false,
  "values": {
    "base_gen": {
      "injected": 0.25,
      "control": 0.25
    },
    "base_probe": {
      "injected": 0.375,
      "control": 0.1875
    },
    "sft_gen": {
      "injected": 0.125,
      "control": 0.0625
    },
    "sft_probe": {
      "injected": 0.3125,
      "control": 0.0625
    }
  },
  "probe_gain_injected": -0.0625,
  "probe_change_control": -0.125,
  "generated_change_control": -0.1875,
  "format_rate": 0.0,
  "thresholds": {
    "min_generated_injected": 0.75,
    "min_probe_gain": 0.25,
    "max_abs_control_change": 0.125,
    "min_format_rate": 0.9
  },
  "dataset_sha256": "a9d359dfe03251b0b08e6d25f22870348dc900a7d4c505cb3b341a1b92ee1583",
  "start_record": "/content/drive/MyDrive/inm363/results/034_smoke/start_seed0.json",
  "sft_config": "configs/034_smoke_sft_colab_b15.yaml"
}


AssertionError: LoRA qualification failed, stop before the GRPO smoke

## 12. GRPO smoke branch


In [ ]:
grpo_status = train_smoke('grpo', GRPO_CONFIG, GRPO_TAG, GRPO_ADAPTER, GRPO_RESULT)
evaluate_smoke('grpo', GRPO_CONFIG, GRPO_TAG, GRPO_ADAPTER, GRPO_RESULT)
print('GRPO wall seconds:', grpo_status.get('wall_seconds'))


## 13. Smoke summary


In [ ]:
def window_mean(values, n=10):
    return sum(values[:n]) / min(n, len(values)) if values else None

def last_window_mean(values, n=10):
    return sum(values[-n:]) / min(n, len(values)) if values else None

def eval_items_per_second(result_dir, branch):
    manifest = read_json(os.path.join(result_dir, f'{branch}_eval_manifest.json'))
    seconds = manifest['worker_status']['wall_seconds']
    return manifest['n_items'] / seconds

sft_log = read_json(os.path.join(SFT_RESULT, 'sft_trainlog.json'))
grpo_log = read_json(os.path.join(GRPO_RESULT, 'grpo_trainlog.json'))
sft_losses = [row['loss'] for row in sft_log if row.get('loss') is not None]
lengths = [row['completions/mean_length'] for row in grpo_log
           if row.get('completions/mean_length') is not None]
accuracy_rewards = [row['rewards/accuracy_reward/mean'] for row in grpo_log
                    if row.get('rewards/accuracy_reward/mean') is not None]
format_rewards = [row['rewards/format_reward/mean'] for row in grpo_log
                  if row.get('rewards/format_reward/mean') is not None]
reward_spread = [row['reward_std'] for row in grpo_log
                 if row.get('reward_std') is not None]
summary = {
    'completed': True,
    'dataset_sha256': smoke['sha256'],
    'sft_qualification': qualification,
    'sft_steps': SFT_CFG.train.max_steps,
    'grpo_steps': GRPO_CFG.train.max_steps,
    'completion_cap': GRPO_CFG.grpo.max_completion_length,
    'sft_wall_seconds': sft_status['wall_seconds'],
    'grpo_wall_seconds': grpo_status['wall_seconds'],
    'sft_seconds_per_step': sft_status['wall_seconds'] / SFT_CFG.train.max_steps,
    'grpo_seconds_per_step': grpo_status['wall_seconds'] / GRPO_CFG.train.max_steps,
    'mean_grpo_completion_tokens': sum(lengths) / len(lengths) if lengths else None,
    'mean_generated_tokens_per_grpo_step': (
        sum(lengths) / len(lengths) * GRPO_CFG.train.num_generations if lengths else None),
    'evaluation_items_per_second': {
        'baseline': eval_items_per_second(BASE_RESULT, 'baseline'),
        'sft': eval_items_per_second(SFT_RESULT, 'sft'),
        'grpo': eval_items_per_second(GRPO_RESULT, 'grpo'),
    },
    'sft_loss_first10': window_mean(sft_losses),
    'sft_loss_last10': last_window_mean(sft_losses),
    'grpo_accuracy_reward_first10': window_mean(accuracy_rewards),
    'grpo_accuracy_reward_last10': last_window_mean(accuracy_rewards),
    'grpo_format_reward_first10': window_mean(format_rewards),
    'grpo_format_reward_last10': last_window_mean(format_rewards),
    'grpo_mean_reward_std': sum(reward_spread) / len(reward_spread) if reward_spread else None,
}
summary_path = os.path.join(SMOKE_ROOT, 'smoke_summary.json')
if os.path.exists(summary_path):
    assert read_json(summary_path) == summary, 'smoke summary changed on rerun'
else:
    atomic_write_json(summary_path, summary)
print(json.dumps(summary, indent=2))
print('Use this summary to fix the real W2 budgets before building the six arms.')


## 14. Close the units record


In [14]:
import json, os, time
cu_after = 517.63
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run section 0 first')
else:
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            record = json.load(f)
        record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        record['colab_cu_after'] = cu_after
        record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
        record['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(record) + '\n')
        os.remove(open_path)
        print('logged:', record)


logged: {'session_id': '20260815_190425', 'ts_start': '2026-08-15 19:07', 'notebook': '034_colab_w2_smoke', 'colab_cu_before': 522.26, 'unit_rate': 6.77, 'ts_end': '2026-08-15 19:51', 'colab_cu_after': 517.63, 'colab_cu_used': 4.63, 'closed_on_resume': False}
